# Yuk Pahami Histogram Equalization (HE)

Pernah lihat foto yang kelihatan pudar, berkabut, atau "datar" — nggak ada yang benar-benar hitam pekat atau putih terang? Itu biasanya karena nilai-nilai pikselnya menumpuk di area sempit. Nah, **Histogram Equalization** adalah salah satu cara paling klasik untuk memperbaiki itu.

Di notebook ini kita nggak cuma manggil satu fungsi terus selesai. Kita akan **bikin sendiri** HE dari nol pakai NumPy, biar kelihatan jelas apa yang sebenarnya terjadi di balik layar. Setelah itu baru kita bandingkan dengan fungsi bawaan OpenCV untuk mengecek apakah hasil kita sudah benar.

Yang akan kita lakukan, urut dari atas ke bawah:
1. Upload foto, lihat bentuknya dan histogramnya
2. Bikin HE manual, langkah per langkah
3. Cek hasilnya sama nggak sama fungsi bawaan OpenCV
4. Lihat grafiknya biar makin kebayang
5. Latihan kecil buat mempertajam pemahaman (dan bahan laporan)

Santai aja, jalankan sel-nya satu-satu dari atas. Bisa langsung dijalankan di Google Colab tanpa install apa-apa.

## 1. Pilih foto kamu

Silakan upload foto apa saja. Tapi supaya efek HE-nya kelihatan jelas dan nggak bikin bingung, coba cari foto yang agak **pudar atau kontrasnya rendah** — misalnya foto berkabut, foto yang diambil melawan cahaya (backlit), atau foto yang terlihat suram. Kalau fotonya sudah tajam dan kontrasnya bagus dari awal, nanti hasil HE-nya nggak terlalu kelihatan bedanya — dan itu justru hal menarik yang akan kita bahas juga nanti.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Pilih file secara otomatis: upload di Colab, dialog file di laptop.
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Tidak ada file yang di-upload.')
    filename = next(iter(uploaded))
else:
    filename = None
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        filename = filedialog.askopenfilename(
            title='Pilih gambar untuk Histogram Equalization',
            filetypes=[('Gambar', '*.jpg *.jpeg *.png *.bmp *.tif *.tiff'), ('Semua file', '*.*')]
        )
        root.destroy()
    except Exception as error:
        print(f'Dialog file tidak tersedia ({error}).')

    if not filename:
        filename = input('Masukkan path file gambar: ').strip().strip('\"')
    if not filename:
        raise ValueError('Path gambar belum diisi.')
    filename = str(Path(filename).expanduser())

img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)
if img is None:
    raise FileNotFoundError(f'Gambar tidak dapat dibaca: {filename}')
print('File:', Path(filename).resolve())
print('Ukuran citra:', img.shape, '(tinggi, lebar)')
print('Nilai piksel paling gelap:', img.min(), '| paling terang:', img.max())

plt.figure(figsize=(5,5))
plt.imshow(img, cmap='gray')
plt.title('Foto aslinya (sudah diubah ke grayscale)')
plt.axis('off')
plt.show()

### Coba lihat dulu histogramnya

Histogram itu cuma grafik yang menghitung: dari nilai keabuan 0 (hitam) sampai 255 (putih), masing-masing dipakai oleh berapa piksel. Kalau batang-batangnya numpuk semua di satu area sempit (misalnya cuma di rentang 60-140), itu tandanya fotonya memang kurang kontras — kandidat bagus buat dicoba HE.

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(img.ravel(), bins=256, range=(0,255), color='gray')
plt.title('Histogram foto asli')
plt.xlabel('Nilai piksel (0 = hitam, 255 = putih)')
plt.ylabel('Jumlah piksel')
plt.xlim(0,255)
plt.show()

print(f'Rata-rata kecerahan : {img.mean():.2f}')
print(f'Sebaran nilai (std) : {img.std():.2f}  -> makin besar angkanya, makin kontras fotonya')
print(f'Rentang yang dipakai: dari {img.min()} sampai {img.max()} (dari total kemungkinan 0-255)')

## 2. Sekarang kita bikin HE-nya sendiri, dari nol

HE itu sebenarnya cuma 4 langkah sederhana. Coba kita bedah satu-satu dulu sebelum lihat kodenya, biar nggak kerasa kayak sihir.

**Langkah 1 — Hitung histogram.** Ini sama seperti yang barusan kita bikin: buat setiap nilai keabuan dari 0 sampai 255, hitung ada berapa piksel yang punya nilai itu persis.

**Langkah 2 — Hitung CDF (jumlah kumulatif).** Bayangkan kamu jalan dari nilai 0 ke 255 sambil bawa "keranjang" — setiap lewat satu nilai, kamu tambahkan jumlah piksel di nilai itu ke keranjangmu, tanpa pernah dikurangi. Jadi `cdf(i)` itu jawaban dari pertanyaan: *"sampai nilai i ini, sudah berapa banyak piksel yang terhitung?"*

**Langkah 3 — Ubah CDF jadi rentang 0-255 lagi.** Ini bagian intinya:

$$\text{baru}(i) = \text{round}\left(\frac{cdf(i) - cdf_{min}}{N - cdf_{min}} \times 255\right)$$

Kedengarannya rumit, tapi maksudnya sederhana: hitung dulu nilai `i` ini ada di posisi berapa persen kalau semua piksel diurutkan dari paling gelap (itu bagian pembagian), terus ubah persentase itu jadi skala 0-255 lagi (itu bagian dikali 255). `N` adalah total semua piksel di foto, dan `cdf_min` itu nilai CDF tak-nol yang paling kecil — supaya piksel paling gelap yang benar-benar ada di fotomu jatuh tepat ke 0, bukan ke angka kecil lain.

**Langkah 4 — Ganti setiap piksel pakai tabel dari langkah 3.** Tinggal tukar, nilai lama diganti nilai baru sesuai hasil langkah 3.

In [ ]:
def histogram_equalization_manual(gray_img):
    """HE dari nol, mengikuti 4 langkah yang barusan kita bahas."""

    # Langkah 1: hitung histogram
    hist, _ = np.histogram(gray_img.ravel(), bins=256, range=(0,256))

    # Langkah 2: hitung CDF (jumlah kumulatif dari histogram)
    cdf = hist.cumsum()

    # Langkah 3: ubah CDF jadi rentang 0-255
    cdf_min = cdf[cdf > 0].min()   # nilai cdf tak-nol paling kecil
    N = gray_img.size              # total jumlah piksel di foto
    # Gambar seragam memiliki N == cdf_min; tidak perlu diubah.
    if N == cdf_min:
        cdf_normalized = np.arange(256, dtype=np.uint8)
    else:
        cdf_normalized = np.round((cdf - cdf_min) / (N - cdf_min) * 255)
        cdf_normalized = np.clip(cdf_normalized, 0, 255).astype(np.uint8)

    # Langkah 4: tukar setiap piksel pakai tabel yang baru dihitung
    equalized = cdf_normalized[gray_img]

    return equalized, hist, cdf, cdf_normalized

he_manual, hist_orig, cdf_orig, mapping_table = histogram_equalization_manual(img)
print('Selesai! Ukuran hasilnya:', he_manual.shape, '| tipe datanya:', he_manual.dtype)

## 3. Coba dicek, bener nggak sih hasilnya?

Cara paling gampang mengecek kode HE manual kita bener atau nggak: bandingkan sama fungsi bawaan OpenCV, `cv2.equalizeHist()`. Kalau logikanya sama, hasilnya seharusnya identik (atau beda tipis banget, cuma karena pembulatan).

In [ ]:
he_opencv = cv2.equalizeHist(img)

selisih = np.abs(he_manual.astype(int) - he_opencv.astype(int))
print(f'Rata-rata selisih piksel (manual vs OpenCV): {selisih.mean():.4f}')
print(f'Selisih paling besar                       : {selisih.max()}')
print('(Kalau angkanya 0 atau mendekati 0, berarti kode manual kita sudah benar)')

fig, ax = plt.subplots(1, 3, figsize=(13,5))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('Foto asli'); ax[0].axis('off')
ax[1].imshow(he_manual, cmap='gray'); ax[1].set_title('HE hasil kode kita'); ax[1].axis('off')
ax[2].imshow(he_opencv, cmap='gray'); ax[2].set_title('HE dari cv2.equalizeHist'); ax[2].axis('off')
plt.tight_layout()
plt.show()

## 4. Lihat prosesnya dalam bentuk grafik

Tiga grafik ini bakal bikin kamu "ngeh" kenapa HE bisa nambah kontras — bukan cuma percaya rumus, tapi lihat sendiri buktinya. Ini juga bagus banget buat dilampirkan di laporan tugas.

In [ ]:
hist_he, _ = np.histogram(he_manual.ravel(), bins=256, range=(0,256))

fig, ax = plt.subplots(1, 3, figsize=(16,4.5))

# (a) Histogram sebelum vs sesudah
ax[0].hist(img.ravel(), bins=256, range=(0,255), alpha=0.5, label='Sebelum', color='gray')
ax[0].hist(he_manual.ravel(), bins=256, range=(0,255), alpha=0.5, label='Sesudah HE', color='tab:orange')
ax[0].set_title('(a) Histogram: sebelum vs sesudah')
ax[0].set_xlabel('Nilai piksel'); ax[0].legend()

# (b) Kurva CDF sebelum vs sesudah
ax[1].plot(cdf_orig / cdf_orig.max(), label='CDF sebelum', color='gray')
cdf_he = hist_he.cumsum()
ax[1].plot(cdf_he / cdf_he.max(), label='CDF sesudah HE', color='tab:orange')
ax[1].plot([0,255],[0,1], '--', color='lightgray', label='garis lurus (ideal)')
ax[1].set_title('(b) Kurva CDF (diskalakan 0-1)')
ax[1].set_xlabel('Nilai piksel'); ax[1].legend()

# (c) Tabel penukaran nilai lama -> nilai baru
ax[2].plot(np.arange(256), mapping_table, color='tab:blue')
ax[2].plot([0,255],[0,255], '--', color='lightgray', label='kalau nggak berubah sama sekali')
ax[2].set_title('(c) Tabel penukaran nilai HE')
ax[2].set_xlabel('Nilai piksel lama'); ax[2].set_ylabel('Nilai piksel baru')
ax[2].legend()

plt.tight_layout()
plt.show()

print('Cara baca grafik (b): makin curam/tegak kurva CDF di suatu area, artinya makin banyak')
print('piksel menumpuk di area nilai itu. HE membuat kurva CDF mendekati garis lurus diagonal')
print('-- semakin lurus kurvanya, semakin "merata" histogram hasilnya.')

## 5. Bandingkan angkanya, biar nggak cuma "kelihatan lebih bagus"

Bilang "fotonya jadi lebih jelas" itu subjektif. Biar laporanmu lebih kuat, kasih juga angka yang bisa dibandingkan.

In [ ]:
def entropy(gray_img):
    hist, _ = np.histogram(gray_img.ravel(), bins=256, range=(0,256))
    p = hist / hist.sum()
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

print(f"{'Ukuran':<22}{'Sebelum':>12}{'Sesudah HE':>14}")
print('-'*48)
print(f"{'Rata-rata kecerahan':<22}{img.mean():>12.2f}{he_manual.mean():>14.2f}")
print(f"{'Sebaran nilai (std)':<22}{img.std():>12.2f}{he_manual.std():>14.2f}")
print(f"{'Entropy (bit)':<22}{entropy(img):>12.3f}{entropy(he_manual):>14.3f}")
print(f"{'Rentang yang dipakai':<22}{f'{img.min()}-{img.max()}':>12}{f'{he_manual.min()}-{he_manual.max()}':>14}")

print()
print('Singkatnya: std yang naik berarti kontras naik, entropy yang naik berarti')
print('informasi/detail yang bisa dibedakan mata juga bertambah.')

## 6. Sekarang giliran kamu coba-coba

Ini bukan sekadar "jalankan kode terus selesai" — coba benar-benar amati hasilnya dan tulis apa yang kamu lihat. Ini juga bahan yang pas banget buat bagian pembahasan di laporan tugas kamu.

1. **Coba foto lain yang kontrasnya udah bagus dari awal.** Apa yang berubah dari hasil HE-nya kali ini? Masih membantu, atau malah nggak ada bedanya (bahkan mungkin bikin noise-nya makin kelihatan)? Coba jelaskan pakai kurva CDF-nya kenapa bisa begitu.
2. **Coba juga pada foto berwarna.** Buka pakai `cv2.IMREAD_COLOR`, lalu terapkan HE **cuma di channel V** dari ruang warna HSV (bukan langsung ke R, G, B satu-satu — coba pikirkan dulu, kenapa kalau HE diterapkan terpisah ke R, G, B malah bisa merusak warnanya?).
3. **Coba print nilai `cdf_min`** dari fotomu. Menurutmu itu angka apa, dan kenapa harus dikurangkan di rumus?
4. **Hitung entropy** untuk beberapa foto yang berbeda-beda, sebelum dan sesudah HE. Apakah entropy-nya selalu naik? Kalau ternyata nggak selalu, kira-kira kapan itu bisa terjadi?
5. Coba tulis 2-3 kalimat sendiri: **kapan sebaiknya pakai HE**, dan **kapan sebaiknya jangan** (kalau kamu sudah belajar CLAHE juga, boleh dibandingkan).

In [ ]:
# Ruang kerja buat latihan nomor 2: HE pada foto berwarna lewat channel V (HSV)
# Hapus tanda pagar (#) di bawah ini setelah upload foto berwarna baru kalau mau coba.

# img_color = cv2.imread(filename)                          # OpenCV baca dalam format BGR
# hsv = cv2.cvtColor(img_color, cv2.COLOR_BGR2HSV)
# h, s, v = cv2.split(hsv)
# v_eq = cv2.equalizeHist(v)
# hsv_eq = cv2.merge([h, s, v_eq])
# result_color = cv2.cvtColor(hsv_eq, cv2.COLOR_HSV2BGR)
#
# plt.figure(figsize=(10,5))
# plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)); plt.title('Sebelum'); plt.axis('off')
# plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(result_color, cv2.COLOR_BGR2RGB)); plt.title('Sesudah (HE di channel V)'); plt.axis('off')
# plt.show()

## Yang perlu kamu ingat dari notebook ini

- HE bekerja dengan meregangkan histogram — caranya, mengambil kurva CDF dari histogram, lalu memakai kurva itu langsung sebagai "tabel tukar" nilai piksel.
- Kode manual kita (histogram → CDF → normalisasi → tukar) menghasilkan output yang sama persis dengan `cv2.equalizeHist()` — jadi sekarang kamu tahu persis apa yang dikerjakan fungsi itu di baliknya, nggak lagi jadi kotak hitam.
- HE itu sifatnya **global** — satu tabel tukar dipakai rata ke semua piksel, nggak peduli lokasinya di foto sebelah mana.
- Jangan cuma bilang "kelihatan lebih bagus" — angka seperti std dan entropy bikin klaimmu soal peningkatan kontras jadi lebih bisa dipertanggungjawabkan di laporan.